# 📊 EDA & Analysis — AI Predictive Maintenance
This notebook explores the synthetic IoT sensor dataset and validates our feature engineering.

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils import load_config
from src.data_loader import load_or_generate_data
from src.preprocess import run_preprocessing, get_feature_columns

sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 5)
config = load_config('../config/config.yaml')
print('Config loaded ✅')

## 1. Load Dataset

In [ ]:
raw_df = load_or_generate_data(config)
print(f'Shape: {raw_df.shape}')
print(f'Failure rate: {raw_df["failure"].mean():.2%}')
raw_df.head(10)

## 2. Basic Statistics

In [ ]:
raw_df.describe().round(2)

## 3. Class Balance

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
raw_df['failure'].value_counts().plot(kind='bar', ax=ax[0],
    color=['#1E88E5', '#E53935'], edgecolor='white')
ax[0].set_title('Class Distribution')
ax[0].set_xticklabels(['No Failure', 'Failure'], rotation=0)

raw_df['failure'].value_counts().plot(kind='pie', ax=ax[1],
    labels=['No Failure', 'Failure'],
    colors=['#1E88E5', '#E53935'], autopct='%1.1f%%')
ax[1].set_title('Failure Rate')
plt.tight_layout()
plt.show()

## 4. Sensor Distributions by Class

In [ ]:
sensors = ['temperature', 'vibration', 'current', 'pressure']
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for ax, sensor in zip(axes, sensors):
    for label, color in [(0, '#1E88E5'), (1, '#E53935')]:
        subset = raw_df[raw_df['failure'] == label][sensor]
        ax.hist(subset, bins=40, alpha=0.6, color=color,
                label='Failure' if label else 'Normal', density=True)
    ax.set_title(f'{sensor.capitalize()} Distribution', fontsize=12)
    ax.legend()
    ax.set_xlabel(sensor)

plt.suptitle('Sensor Distributions: Normal vs Failure', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Correlation Heatmap

In [ ]:
corr = raw_df[sensors + ['failure']].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, square=True)
plt.title('Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Feature-Engineered Dataset

In [ ]:
proc_df = run_preprocessing(raw_df, config)
feat_cols = get_feature_columns(proc_df)
print(f'Total features after engineering: {len(feat_cols)}')
print('Features:', feat_cols[:10], '...')
proc_df.head(5)

## 7. Stress Score vs Failure

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for label, color, name in [(0, '#1E88E5', 'Normal'), (1, '#E53935', 'Failure')]:
    subset = proc_df[proc_df['failure'] == label]['stress_score']
    ax.hist(subset, bins=40, alpha=0.7, color=color,
            label=name, density=True)
ax.set_title('Composite Stress Score Distribution', fontsize=13)
ax.set_xlabel('Stress Score')
ax.legend()
plt.tight_layout()
plt.show()

## 8. Time-Series Snippet (First 500 samples)

In [ ]:
snippet = raw_df.iloc[:500]
failure_pts = snippet[snippet['failure'] == 1]

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(snippet.index, snippet['temperature'], color='#FF7043',
        linewidth=0.8, label='Temperature')
ax.scatter(failure_pts.index, failure_pts['temperature'],
           color='red', s=40, zorder=5, label='Failure Event')
ax.set_title('Temperature Signal — First 500 Samples', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()
print(f'Failure events in snippet: {len(failure_pts)}')